In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
# Team code standardization mapping
HIST_TO_MODERN = {
    # Relocations / renames
    "SEA": "OKC",   # SuperSonics -> Thunder
    "VAN": "MEM",   # Vancouver Grizzlies -> Memphis
    "NJN": "BKN",   # New Jersey Nets -> Brooklyn
    "BRK": "BKN",   # Brooklyn alt code
    "NOH": "NOP",   # New Orleans Hornets -> Pelicans
    "NOK": "NOP",   # New Orleans/Oklahoma City Hornets -> Pelicans

    # Charlotte franchise codes across eras/data products
    "CHH": "CHA",   # Charlotte Hornets (old)
    "CHO": "CHA",   # Charlotte Hornets (common alt)
    "CHA": "CHA",

    # Washington historical codes
    "WSB": "WAS",   # Washington Bullets -> Wizards
    "BAL": "WAS",   # Baltimore Bullets (older datasets)
    "WAS": "WAS",

    # Common abbreviation variants you might see
    "PHO": "PHX",
    "GS":  "GSW",
    "SA":  "SAS",
    "NY":  "NYK",
    "BK":  "BKN",

    # Modern teams that generally stay the same
    "ATL": "ATL", "BOS": "BOS", "CHI": "CHI", "CLE": "CLE", "DAL": "DAL",
    "DEN": "DEN", "DET": "DET", "GSW": "GSW", "HOU": "HOU", "IND": "IND",
    "LAC": "LAC", "LAL": "LAL", "MEM": "MEM", "MIA": "MIA", "MIL": "MIL",
    "MIN": "MIN", "NOP": "NOP", "NYK": "NYK", "OKC": "OKC", "ORL": "ORL",
    "PHI": "PHI", "PHX": "PHX", "POR": "POR", "SAC": "SAC", "SAS": "SAS",
    "TOR": "TOR", "UTA": "UTA",
}

def canonical_team_abbrev(abbrev: str) -> str:
    """Convert any team abbreviation to its modern canonical form."""
    if abbrev is None or pd.isna(abbrev):
        return None
    a = str(abbrev).strip().upper()
    return HIST_TO_MODERN.get(a, a)  # unknowns pass through so you can detect them

print(f"Team mapping defined with {len(set(HIST_TO_MODERN.values()))} modern teams")
print(f"Mapping {len(HIST_TO_MODERN)} codes total")

Team mapping defined with 30 modern teams
Mapping 44 codes total


In [ ]:
# Standardize schedule CSVs
print("Standardizing schedule files...")
schedule_files = glob.glob("data/schedules/schedule_*.csv")

for sched_file in schedule_files:
    df = pd.read_csv(sched_file, dtype={"GAME_ID": str})
    
    # Parse and standardize matchups
    def standardize_matchup(matchup):
        if pd.isna(matchup):
            return matchup
        
        if " vs. " in matchup:
            home, away = matchup.split(" vs. ")
            home = canonical_team_abbrev(home.strip())
            away = canonical_team_abbrev(away.strip())
            return f"{home} vs. {away}"
        elif " @ " in matchup:
            away, home = matchup.split(" @ ")
            away = canonical_team_abbrev(away.strip())
            home = canonical_team_abbrev(home.strip())
            return f"{away} @ {home}"
        return matchup
    
    df["MATCHUP"] = df["MATCHUP"].apply(standardize_matchup)
    df.to_csv(sched_file, index=False)
    print(f"  ✓ {os.path.basename(sched_file)}")

print("✓ Schedule files standardized")

In [ ]:
# Standardize games.csv
print("Standardizing games.csv...")
games_file = "data/games.csv"
if os.path.exists(games_file):
    df = pd.read_csv(games_file, dtype={"game_id": str})
    
    # Standardize team columns
    if "away_team" in df.columns:
        df["away_team"] = df["away_team"].apply(canonical_team_abbrev)
    if "home_team" in df.columns:
        df["home_team"] = df["home_team"].apply(canonical_team_abbrev)
    
    df.to_csv(games_file, index=False)
    print(f"  ✓ Standardized {len(df)} games")
else:
    print("  ⊘ games.csv not found")

print("✓ games.csv standardized")

In [ ]:
# Standardize players_live files (both filenames and content)
print("Standardizing players_live files...")
player_files = glob.glob("data/players_live/*.csv")
renamed_count = 0
standardized_count = 0

for old_file in player_files:
    filename = os.path.basename(old_file)
    parts = filename.replace('.csv', '').split('_')
    
    if len(parts) >= 3:
        game_id = parts[0]
        team1 = canonical_team_abbrev(parts[1])
        team2 = canonical_team_abbrev(parts[2])
        
        new_filename = f"{game_id}_{team1}_{team2}.csv"
        new_file = os.path.join("data/players_live", new_filename)
        
        # Read and standardize content
        try:
            df = pd.read_csv(old_file, dtype={"personId": str, "PLAYER_ID": str, "teamId": str, "TEAM_ID": str})
            
            # Standardize team ID columns
            if "teamId" in df.columns:
                df["teamId"] = df["teamId"].apply(lambda x: canonical_team_abbrev(str(x)) if not pd.isna(x) else x)
            if "TEAM_ID" in df.columns:
                df["TEAM_ID"] = df["TEAM_ID"].apply(lambda x: canonical_team_abbrev(str(x)) if not pd.isna(x) else x)
            
            # Save with new name
            df.to_csv(new_file, index=False)
            standardized_count += 1
            
            # Remove old file if name changed
            if old_file != new_file:
                os.remove(old_file)
                renamed_count += 1
                
        except Exception as e:
            print(f"  Error processing {filename}: {e}")

print(f"  ✓ Standardized {standardized_count} files")
print(f"  ✓ Renamed {renamed_count} files")
print("✓ players_live files standardized")

In [ ]:
# Standardize games_live files (both filenames and content)
print("Standardizing games_live files...")
games_live_files = glob.glob("data/games_live/*.csv")
renamed_count = 0

for old_file in games_live_files:
    filename = os.path.basename(old_file)
    parts = filename.replace('.csv', '').split('_')
    
    if len(parts) >= 3:
        date = parts[0]
        team1 = canonical_team_abbrev(parts[1])
        team2 = canonical_team_abbrev(parts[2])
        
        new_filename = f"{date}_{team1}_{team2}.csv"
        new_file = os.path.join("data/games_live", new_filename)
        
        # Rename if needed
        if old_file != new_file:
            os.rename(old_file, new_file)
            renamed_count += 1

print(f"  ✓ Renamed {renamed_count} files")
print("✓ games_live files standardized")

In [ ]:
# Standardize kalshi_live files (both filenames and content if applicable)
print("Standardizing kalshi_live files...")
kalshi_files = glob.glob("data/kalshi_live/*.csv")
renamed_count = 0

for old_file in kalshi_files:
    filename = os.path.basename(old_file)
    # Format: YYYYMMDD_TEAM1_TEAM2_kalshi_100ms.csv
    parts = filename.replace('_kalshi_100ms.csv', '').split('_')
    
    if len(parts) >= 3:
        date = parts[0]
        team1 = canonical_team_abbrev(parts[1])
        team2 = canonical_team_abbrev(parts[2])
        
        new_filename = f"{date}_{team1}_{team2}_kalshi_100ms.csv"
        new_file = os.path.join("data/kalshi_live", new_filename)
        
        # Rename if needed
        if old_file != new_file:
            os.rename(old_file, new_file)
            renamed_count += 1

print(f"  ✓ Renamed {renamed_count} files")
print("✓ kalshi_live files standardized")

In [ ]:
# Verify standardization - count unique teams
print("\n" + "="*60)
print("Verification: Counting unique teams after standardization")
print("="*60)

# Check schedules
schedule_teams = set()
for sched_file in glob.glob("data/schedules/schedule_*.csv"):
    df = pd.read_csv(sched_file)
    for matchup in df["MATCHUP"]:
        if " vs. " in str(matchup):
            teams = matchup.split(" vs. ")
        elif " @ " in str(matchup):
            teams = matchup.split(" @ ")
        else:
            continue
        for team in teams:
            schedule_teams.add(team.strip())

print(f"\nSchedules: {len(schedule_teams)} unique teams")
print(f"  {sorted(schedule_teams)}")

# Check games.csv
if os.path.exists("data/games.csv"):
    games_df = pd.read_csv("data/games.csv")
    games_teams = set()
    if "away_team" in games_df.columns:
        games_teams.update(games_df["away_team"].dropna().unique())
    if "home_team" in games_df.columns:
        games_teams.update(games_df["home_team"].dropna().unique())
    print(f"\ngames.csv: {len(games_teams)} unique teams")

print(f"\n✓ Standardization complete!")

In [ ]:
# Load all schedule files to create game_id -> game_date mapping
schedule_files = glob.glob("data/schedules/schedule_*.csv")
print(f"Loading {len(schedule_files)} schedule files...")

game_date_map = {}
for sched_file in schedule_files:
    df = pd.read_csv(sched_file, dtype={"GAME_ID": str})
    for _, row in df.iterrows():
        game_id = row["GAME_ID"].zfill(10)
        game_date_map[game_id] = row["GAME_DATE"]

print(f"Loaded {len(game_date_map)} games with dates")
print(f"Sample: {list(game_date_map.items())[:3]}")